# Quant Vision: Deciphering Stock Patterns
### Brain and Cognitive Science Club — Winter Project 2025

Reconstructed notebook following the final project report. Pipeline:

1. Pull OHLC data for 10 tickers (2018–2023 train, Jan–Jun 2024 test)
2. Label each day as **Hammer**, **Doji**, or **None** from candle anatomy
3. Convert 10-day windows into normalized 64×64 RGB candlestick images
4. Train a multi-task CNN (pattern classification + next-day direction) on the images
5. Evaluate pattern classification performance and run a simple trading simulation


In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, optimizers, losses
from tensorflow.keras.utils import to_categorical

from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
tf.random.set_seed(42)


## 3.1 Data Collection: Getting the Raw Numbers

Daily OHLC data for ten major tickers via `yfinance`. Split chronologically (not randomly)
to avoid look-ahead bias.

- **Train:** Jan 2018 – Dec 2023
- **Test:** Jan 2024 – Jun 2024


In [ ]:
TICKERS = ["AAPL", "GOOGL", "MSFT", "AMZN", "TSLA", "NVDA", "AMD", "INTC", "NFLX", "META"]

TRAIN_START = "2018-01-01"
TRAIN_END   = "2023-12-31"
TEST_START  = "2024-01-01"
TEST_END    = "2024-06-30"

def download_ticker_data(ticker, start, end):
    df = yf.download(ticker, start=start, end=end, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df[["Open", "High", "Low", "Close"]].dropna()
    df["Ticker"] = ticker
    return df

raw_data = {}
for ticker in TICKERS:
    print(f"Downloading {ticker}...")
    raw_data[ticker] = download_ticker_data(ticker, TRAIN_START, TEST_END)

raw_data[TICKERS[0]].head()


## 3.2 Pattern Labeling: Defining the Shapes

For every day we compute:

- `Body = |Open - Close|`
- `Lower Wick = min(Open, Close) - Low`
- `Upper Wick = High - max(Open, Close)`
- `Total Range = High - Low`

Labeling rules:

- **Hammer:** `Lower Wick >= 2 * Body` and `Upper Wick <= 0.5 * Body`
- **Doji:** `Body < 0.10 * Total Range`
- **None:** anything else


In [ ]:
def compute_candle_metrics(df):
    df = df.copy()
    df["Body"] = (df["Open"] - df["Close"]).abs()
    df["LowerWick"] = df[["Open", "Close"]].min(axis=1) - df["Low"]
    df["UpperWick"] = df["High"] - df[["Open", "Close"]].max(axis=1)
    df["TotalRange"] = df["High"] - df["Low"]
    return df

def label_pattern(row):
    body, lower_wick, upper_wick, total_range = row["Body"], row["LowerWick"], row["UpperWick"], row["TotalRange"]

    if total_range == 0:
        return "None"

    # Hammer: tiny body, long lower wick, small upper wick
    if lower_wick >= 2 * body and upper_wick <= 0.5 * body:
        return "Hammer"

    # Doji: market indecision, body is a sliver of the full range
    if body < 0.10 * total_range:
        return "Doji"

    return "None"

def label_dataframe(df):
    df = compute_candle_metrics(df)
    df["Pattern"] = df.apply(label_pattern, axis=1)
    df["NextClose"] = df["Close"].shift(-1)
    df["Direction"] = np.where(df["NextClose"] > df["Close"], "Up", "Down")
    return df

labeled_data = {ticker: label_dataframe(df) for ticker, df in raw_data.items()}
labeled_data[TICKERS[0]]["Pattern"].value_counts()


## 3.3 Image Generation: Turning Numbers into Pictures

Each 10-day window is normalized to a 0–1 scale (so the model focuses on shape, not the
price tag) and rendered as a 64×64 RGB candlestick chart with `matplotlib`, using the
standard convention: **green** for up days, **red** for down days.

**Class imbalance:** Hammer and Doji are naturally rare, so 85% of `None` samples are
discarded during dataset construction to stop the model from lazily predicting `None` every time.


In [ ]:
WINDOW_SIZE = 10
IMG_SIZE = 64

def render_candlestick_image(window_df, img_size=IMG_SIZE):
    """Converts a 10-day OHLC window into a 64x64 RGB candlestick image."""
    o = window_df["Open"].values
    h = window_df["High"].values
    l = window_df["Low"].values
    c = window_df["Close"].values

    lo, hi = l.min(), h.max()
    rng = hi - lo if hi > lo else 1e-6
    o_n, h_n, l_n, c_n = (o - lo) / rng, (h - lo) / rng, (l - lo) / rng, (c - lo) / rng

    fig = plt.figure(figsize=(1, 1), dpi=img_size)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.axis("off")
    ax.set_xlim(-0.5, len(window_df) - 0.5)
    ax.set_ylim(0, 1)

    for i in range(len(window_df)):
        color = "green" if c_n[i] >= o_n[i] else "red"
        ax.add_line(Line2D([i, i], [l_n[i], h_n[i]], color=color, linewidth=1))
        body_low, body_high = min(o_n[i], c_n[i]), max(o_n[i], c_n[i])
        body_height = max(body_high - body_low, 0.01)
        ax.add_patch(Rectangle((i - 0.3, body_low), 0.6, body_height, color=color))

    fig.canvas.draw()
    img_array = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
    img_array = img_array.reshape(fig.canvas.get_width_height()[::-1] + (4,))[:, :, :3]
    plt.close(fig)

    img = Image.fromarray(img_array).resize((img_size, img_size))
    return np.array(img) / 255.0

# quick sanity check on one window
sample_window = labeled_data[TICKERS[0]].iloc[0:WINDOW_SIZE]
plt.imshow(render_candlestick_image(sample_window))
plt.axis("off")
plt.title("Sample candlestick image")
plt.show()


In [ ]:
PATTERN_CLASSES = ["Hammer", "Doji", "None"]
DIRECTION_CLASSES = ["Down", "Up"]

def build_windows(df, ticker):
    samples = []
    df = df.reset_index()  # keep Date as a column
    date_col = "Date" if "Date" in df.columns else df.columns[0]
    for i in range(WINDOW_SIZE, len(df) - 1):
        window = df.iloc[i - WINDOW_SIZE:i]
        label_row = df.iloc[i]
        if pd.isna(label_row["Direction"]):
            continue
        samples.append({
            "ticker": ticker,
            "date": str(label_row[date_col].date()) if hasattr(label_row[date_col], "date") else str(label_row[date_col]),
            "window": window,
            "pattern": label_row["Pattern"],
            "direction": label_row["Direction"]
        })
    return samples

all_samples = []
for ticker, df in labeled_data.items():
    all_samples.extend(build_windows(df, ticker))

print(f"Total candidate windows: {len(all_samples)}")

# Handle class imbalance: drop 85% of "None" pattern samples
none_samples = [s for s in all_samples if s["pattern"] == "None"]
pattern_samples = [s for s in all_samples if s["pattern"] != "None"]

np.random.shuffle(none_samples)
keep_none = none_samples[: int(len(none_samples) * 0.15)]

balanced_samples = pattern_samples + keep_none
np.random.shuffle(balanced_samples)

print(f"Hammer/Doji samples: {len(pattern_samples)}")
print(f"None samples kept (15%): {len(keep_none)}")
print(f"Total samples after rebalancing: {len(balanced_samples)}")


In [ ]:
def samples_to_arrays(samples):
    X_img, y_pattern, y_direction = [], [], []
    for s in samples:
        X_img.append(render_candlestick_image(s["window"]))
        y_pattern.append(PATTERN_CLASSES.index(s["pattern"]))
        y_direction.append(DIRECTION_CLASSES.index(s["direction"]))
    X_img = np.array(X_img, dtype=np.float32)
    y_pattern = to_categorical(y_pattern, num_classes=len(PATTERN_CLASSES))
    y_direction = to_categorical(y_direction, num_classes=len(DIRECTION_CLASSES))
    return X_img, y_pattern, y_direction

train_samples = [s for s in balanced_samples if s["date"] < TEST_START]
test_samples  = [s for s in balanced_samples if s["date"] >= TEST_START]

print(f"Train windows: {len(train_samples)}, Test windows: {len(test_samples)}")

X_train, y_pattern_train, y_dir_train = samples_to_arrays(train_samples)
X_test, y_pattern_test, y_dir_test = samples_to_arrays(test_samples)

print("Train:", X_train.shape, y_pattern_train.shape, y_dir_train.shape)
print("Test:", X_test.shape, y_pattern_test.shape, y_dir_test.shape)


## 4. Model Architecture

Conv(32) → BN → MaxPool → Conv(64) → BN → MaxPool → Conv(128) → BN → Global Average
Pooling → Dense (ReLU, L2 regularized) → Dropout → two softmax heads:

- **Pattern head:** Hammer / Doji / None
- **Direction head:** Up / Down

Global Average Pooling is used instead of Flatten to cut parameter count and curb overfitting.


In [ ]:
def build_quantvision_model(input_shape=(64, 64, 3), num_pattern_classes=3, num_direction_classes=2):
    inputs = layers.Input(shape=input_shape, name="candlestick_image")

    x = layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv1_32")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2_64")(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)

    x = layers.Conv2D(128, (3, 3), padding="same", activation="relu", name="conv3_128")(x)
    x = layers.BatchNormalization()(x)

    x = layers.GlobalAveragePooling2D()(x)

    x = layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)

    pattern_output = layers.Dense(num_pattern_classes, activation="softmax", name="pattern")(x)
    direction_output = layers.Dense(num_direction_classes, activation="softmax", name="direction")(x)

    return models.Model(inputs=inputs, outputs=[pattern_output, direction_output], name="QuantVision")

model = build_quantvision_model()
model.summary()


## Compile: Multi-Task Loss Setup

- Optimizer: Adam, learning rate 0.001
- Categorical cross-entropy on both heads
- **Loss weighting:** pattern head weighted higher to anchor the shared features
- **Label smoothing** applied only to the pattern head, for robustness against the
  rare-class noise


In [ ]:
pattern_loss = losses.CategoricalCrossentropy(label_smoothing=0.1)
direction_loss = losses.CategoricalCrossentropy()

model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss={"pattern": pattern_loss, "direction": direction_loss},
    loss_weights={"pattern": 0.7, "direction": 0.3},
    metrics={"pattern": "accuracy", "direction": "accuracy"}
)


## 5. Experimentation and Training

This is Phase 3 (multi-task) of the report directly — Phases 1 (numerical FNN baseline)
and 2 (single-task CNN) are described in the report but aren't reproduced here since they
were superseded.


In [ ]:
EPOCHS = 100
BATCH_SIZE = 32

history = model.fit(
    X_train,
    {"pattern": y_pattern_train, "direction": y_dir_train},
    validation_data=(X_test, {"pattern": y_pattern_test, "direction": y_dir_test}),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)


In [ ]:
# Figure 3: Training and Validation Accuracy over 100 epochs
plt.figure(figsize=(7, 5))
plt.plot(history.history["pattern_accuracy"], label="train_pattern_acc")
plt.plot(history.history["val_pattern_accuracy"], label="val_pattern_acc")
plt.plot(history.history["direction_accuracy"], label="train_direction_acc", linestyle="--")
plt.plot(history.history["val_direction_accuracy"], label="val_direction_acc", linestyle="--")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.show()


## 5.4 Results and Performance Metrics

Classification report and confusion matrix for the pattern head on the held-out 2024 test set
(Table 1 / Figure 2 in the report).


In [ ]:
pattern_pred, direction_pred = model.predict(X_test)
pattern_pred_labels = np.argmax(pattern_pred, axis=1)
pattern_true_labels = np.argmax(y_pattern_test, axis=1)

print(classification_report(pattern_true_labels, pattern_pred_labels, target_names=PATTERN_CLASSES))

cm = confusion_matrix(pattern_true_labels, pattern_pred_labels)
plt.figure(figsize=(5, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="viridis",
            xticklabels=PATTERN_CLASSES, yticklabels=PATTERN_CLASSES)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Pattern Confusion Matrix")
plt.tight_layout()
plt.show()


## 5.5 Simulated Strategy vs. Random Luck

**AI strategy:** enter a trade only when the model predicts `Hammer` + `Up`.
**Random strategy:** enter on a coin-flip.

Reproduces Table 2 in the report.


In [ ]:
direction_pred_labels = np.argmax(direction_pred, axis=1)

def simulate_ai_strategy(test_samples, pattern_pred_labels, direction_pred_labels):
    trades = []
    for i, s in enumerate(test_samples):
        predicted_pattern = PATTERN_CLASSES[pattern_pred_labels[i]]
        predicted_direction = DIRECTION_CLASSES[direction_pred_labels[i]]
        if predicted_pattern == "Hammer" and predicted_direction == "Up":
            actual_direction = s["direction"]
            pnl = 1.0 if actual_direction == "Up" else -1.0
            trades.append(pnl)
    return trades

def simulate_random_strategy(test_samples, seed=42):
    rng = np.random.default_rng(seed)
    trades = []
    for s in test_samples:
        if rng.random() < 0.5:
            actual_direction = s["direction"]
            pnl = 1.0 if actual_direction == "Up" else -1.0
            trades.append(pnl)
    return trades

def summarize_strategy(trades, name):
    trades = np.array(trades)
    win_rate = (trades > 0).mean() * 100 if len(trades) > 0 else 0.0
    total_pnl = trades.sum()
    print(f"{name}: Win Rate={win_rate:.2f}%  Total PnL={total_pnl:.2f}  Total Trades={len(trades)}")
    return win_rate, total_pnl, len(trades)

ai_trades = simulate_ai_strategy(test_samples, pattern_pred_labels, direction_pred_labels)
random_trades = simulate_random_strategy(test_samples)

summarize_strategy(ai_trades, "AI Strategy")
summarize_strategy(random_trades, "Random Strategy")


## 6. Limitations (from the report)

- **Historical bias:** trained only on historical data; markets are non-stationary, so
  sudden regime shifts can break learned patterns.
- **Limited vocabulary:** only Hammer and Doji are modeled; Engulfing, Morning Star, and
  other reversal signals are missing.
- **Data imbalance:** Hammer support is very low (39 in the report's test split), which
  caps recall on that class regardless of architecture tweaks.
- **Idealized backtest:** the trading simulation ignores slippage, fees, and liquidity.
- **Prediction horizon:** the model only forecasts next-day direction, not multi-day trend
  or macro context.
